In [0]:

from pyspark.sql import functions as F
from datetime import date, datetime
import re

# ==================================================
# 1. CONFIGURATION
# ==================================================

dbutils.widgets.text("CATALOG", "vp_vk_mkc20040528")
CATALOG = dbutils.widgets.get("CATALOG")

BRONZE_TABLE = f"{CATALOG}.bronze.members"
SILVER_TABLE = f"{CATALOG}.silver.members"

SNAPSHOT_DATE = date(2026, 4, 1)


# ==================================================
# 2. READ AND STANDARDISE BRONZE
# ==================================================

df = spark.read.table(BRONZE_TABLE)

df = df.select(
    F.upper(F.trim(F.col("member_id"))).alias("member_id"),
    F.initcap(F.trim(F.col("full_name"))).alias("full_name"),
    F.lower(F.trim(F.col("email"))).alias("email"),
    F.regexp_replace(
        F.col("phone").cast("string"), r"\D", ""
    ).alias("phone_raw"),
    F.initcap(F.trim(F.col("city"))).alias("city"),
    F.upper(F.trim(F.col("plan"))).alias("plan"),
    F.coalesce(
        F.try_to_date("joined_on", "yyyy-MM-dd"),
        F.try_to_date("joined_on", "dd/MM/yyyy")
    ).alias("joined_on"),
    F.coalesce(
        F.try_to_date("change_effective_date", "yyyy-MM-dd"),
        F.try_to_date("change_effective_date", "dd/MM/yyyy")
    ).alias("change_effective_date")
)

# Keep only the last 10 digits of phone
df = df.withColumn(
    "phone",
    F.when(
        F.col("phone_raw").isNull() | (F.col("phone_raw") == ""),
        F.lit(None).cast("string")
    ).otherwise(F.expr("right(phone_raw, 10)"))
).drop("phone_raw")

# Remove rows without a member ID
df = df.filter(
    F.col("member_id").isNotNull() & (F.col("member_id") != "")
)

# Identical source records count once
df = df.dropDuplicates([
    "member_id", "full_name", "email", "phone", "city",
    "plan", "joined_on", "change_effective_date"
])

# Collect events in chronological order.
# Suitable for this small assessment dataset.
records = [
    r.asDict(recursive=True)
    for r in df.collect()
]


# ==================================================
# 3. APPLY SCD TYPE 1 + TYPE 2 RULES
# ==================================================

members = {}

# Snapshot rows first, then change rows by effective date
records.sort(
    key=lambda r: (
        r["member_id"],
        r["change_effective_date"] or SNAPSHOT_DATE,
        0 if r["change_effective_date"] is None else 1
    )
)

for r in records:
    mid = r["member_id"]

    event_date = r["change_effective_date"] or SNAPSHOT_DATE

    values = {
        "member_id": mid,
        "full_name": r["full_name"],
        "email": r["email"],
        "phone": r["phone"],
        "city": r["city"],
        "plan": r["plan"],
        "joined_on": r["joined_on"]
    }

    # First occurrence: create initial version
    if mid not in members:
        members[mid] = [{
            **values,
            "effective_from": event_date,
            "effective_to": None,
            "is_current": True
        }]
        continue

    versions = members[mid]
    current = versions[-1]

    # Snapshot rows establish initial state; skip later
    # snapshot duplicates for already-known members.
    if r["change_effective_date"] is None:
        continue

    # SCD Type 2 trigger: plan or city changed
    type2_changed = (
        current["plan"] != values["plan"]
        or current["city"] != values["city"]
    )

    # Type 1 attributes overwrite the current version
    type1_changed = any(
        current[k] != values[k]
        for k in ["full_name", "email", "phone"]
    )

    # No standardised change: do nothing
    if not type2_changed and not type1_changed:
        continue

    if type2_changed:
        # Close current version one day before the new version
        current["effective_to"] = (
            event_date - __import__("datetime").timedelta(days=1)
        )
        current["is_current"] = False

        # New version contains the incoming values
        versions.append({
            **values,
            "effective_from": event_date,
            "effective_to": None,
            "is_current": True
        })

    else:
        # Type 1: overwrite only the current version
        for k in ["full_name", "email", "phone"]:
            current[k] = values[k]


# ==================================================
# 4. CREATE SILVER OUTPUT
# ==================================================

output_rows = [
    version
    for versions in members.values()
    for version in versions
]

schema = """
    member_id STRING,
    full_name STRING,
    email STRING,
    phone STRING,
    city STRING,
    plan STRING,
    joined_on DATE,
    effective_from DATE,
    effective_to DATE,
    is_current BOOLEAN
"""

silver_df = spark.createDataFrame(output_rows, schema=schema)

# Replace output with deterministic rebuilt history.
# Metadata columns are not included.
(
    silver_df
    .select(
        "member_id",
        "full_name",
        "email",
        "phone",
        "city",
        "plan",
        "joined_on",
        "effective_from",
        "effective_to",
        "is_current"
    )
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)